In [14]:
#Instalação
!pip install pymupdf wget pytesseract openpyxl

In [15]:
import fitz
import os

import numpy as np
import pandas as pd

import re

In [16]:
#Loader PDF to Text
class PdfToText():
  def __init__(self):
    pass

  def convert_from_file(self, filename):
    try:
      doc = self.load_pdf(filename)
    except:
      print( "Unable to open file." )
      return ""

    text = ""
    # pagina = f"pagina {i}" + (50 * "-") + "\n\n\n"
    
    i = 0;
    for page in doc:
      text += page.get_text()
      i = i + 1;
    
    doc.close()
    return text

  def load_pdf(self, filename):
    return fitz.open(filename)

class EnemExam(PdfToText):
  def __init__(self):
    pass

  def read_questions(self, filename):
    self.text = self.convert_from_file(filename)
    
    self.find_color()
    self.find_topics()
    self.find_questions()
    

  def find_topics(self):
    """
    Detecta os limites das seções de tópicos com robustez aumentada.
    Se a detecção falhar, usa posição de questão como fallback.
    """
    self._topics = [
        'matemática e suas tecnologias',
        'linguagens, códigos e suas tecnologias',
        'ciências humanas e suas tecnologias',
        'ciências da natureza e suas tecnologias'
    ]

    # Estratégia 1: Procurar pelos cabeçalhos completos
    topic_poss = [self.text.lower().find(topic) for topic in self._topics]
    
    self._topics_loc = [
        (topic, topic_pos) 
        for topic, topic_pos in zip(self._topics, topic_poss) 
        if topic_pos >= 0
    ]
    
    # Se encontrou tópicos, ordena
    if self._topics_loc:
        self._topics_loc.sort(key=lambda t: t[1])
        # Valida se a ordem faz sentido
        self._topics_loc = self._validate_topic_order(self._topics_loc)
    
    # Se não encontrou tópicos suficientes, usará fallback na função define_question_topic
    self._use_position_fallback = len(self._topics_loc) < 2

  def _validate_topic_order(self, topics_loc):
    """
    Valida se a ordem de tópicos faz sentido.
    Se há grandes saltos ou ordem estranha, pode indicar repetição em footers.
    """
    if len(topics_loc) < 2:
        return topics_loc
    
    # Validar que não há repetições muito próximas (< 500 chars)
    # que indicariam header/footer repetido
    validated = [topics_loc[0]]
    
    for i in range(1, len(topics_loc)):
        curr_pos = topics_loc[i][1]
        prev_pos = validated[-1][1]
        
        # Se a diferença é muito pequena, pode ser repetição em rodapé
        if curr_pos - prev_pos < 500:
            continue  # Pular esta ocorrência
        
        validated.append(topics_loc[i])
    
    return validated

  def find_color(self):
    self._color = ""
    padroes = [
        r"CADERNO\s*\n(\w+)",                    # Padrão 1: "CADERNO\nAZUL"
        r"Caderno\s+\d+\s*-\s*(\w+)\s*-"         # Padrão 2: "Caderno 1 - AZUL - ..."
    ]
    
    for padrao in padroes:
      match = re.search(padrao, self.text.lower(), re.IGNORECASE)
      if match:
        self._color = match.group(1).upper()


  def pre_process_text(self, matchs,i):
    processed = re.sub("ENEM \d{4}", "", self.text[matchs[i-1]:matchs[i]])
    processed = re.sub(r"\*[A-Za-z0-9]+\*", "", processed)
    pattern = r"[A-Za-z0-9]{2}\s-\s\d+°\s+dia\s\|\sCaderno\s\d+\s-\s[A-Za-z0-9]{1,8}\s-\s\d+ª\sAplicação\n\s*\d+"
    processed = re.sub(pattern, "", processed)
    return processed

  def find_questions(self):
    matchs = []
    for match in re.finditer('\n(questão) [0-9]+', self.text.lower()):
      matchs.append( match.start() )

    start_pos = matchs[-1]
    match_end = re.search(r'\*[a-z0-9]+\*', self.text.lower()[start_pos:])
    matchs.append(start_pos + match_end.start())
   
    self.questions = []
    self.questions_topic = []
    self.questions_start = []
    for i in range( 1, len(matchs) ):
      self.questions_topic.append( self.define_question_topic(matchs[i-1]) )
      self.questions_start.append( matchs[i-1] )
      self.questions.append(self.pre_process_text(matchs, i))

    
  def define_question_topic(self, start):
    """
    Identifica o tópico (área) de uma questão.
    
    Estratégia 1: Usar posição no PDF (se tópicos foram detectados)
    Estratégia 2: Usar posição da questão como fallback (Q1-45:LC, Q46-90:CH, etc)
    
    Returns: label (0-4)
      0: Desconhecido
      1: Matemática
      2: Linguagens
      3: Humanas
      4: Natureza
    """
    question_topic = 0  # UNKNOWN por padrão
    
    # Estratégia 1: Se tópicos foram detectados no PDF, usar posição
    if self._topics_loc and not self._use_position_fallback:
        for idx, (topic, topic_start) in enumerate(self._topics_loc):
            if start >= topic_start:
                question_topic = idx + 1
            else:
                break
        return min(question_topic, 4)
    
    # Estratégia 2: Fallback - usar posição da questão
    # Estrutura padrão do ENEM Dia 1:
    # Q1-Q45: Linguagens (LC) → label=2
    # Q46-Q90: Ciências Humanas (CH) → label=3
    # Q91-Q135: Ciências da Natureza (CN) → label=4
    # Q136-Q180: Matemática (MT) → label=1
    
    # Extrair número da questão do texto para usar como fallback
    # Isso será feito na célula de processamento com mais contexto
    
    return question_topic


<>:107: SyntaxWarning: invalid escape sequence '\d'
<>:107: SyntaxWarning: invalid escape sequence '\d'
/var/folders/v7/r01_ms3n3bx_lmj8p26f7ctr0000gn/T/ipykernel_59855/3680623166.py:107: SyntaxWarning: invalid escape sequence '\d'
  processed = re.sub("ENEM \d{4}", "", self.text[matchs[i-1]:matchs[i]])


In [17]:
def get_cod_prova_csv(area, cor, df_csv):
    # Mapeia a área para o nome da variável correspondente no CSV
    variaveis = {
        "CH": "CO_PROVA_CH",
        "LC": "CO_PROVA_LC",
        "MT": "CO_PROVA_MT",
        "CN": "CO_PROVA_CN"
    }

    variavel = variaveis.get(area.upper())
    
    if variavel is None:
        return 0  # Área inválida

    # Padroniza a cor para comparação
    cor = cor.strip().upper()
    df_csv["cor"] = df_csv["cor"].astype(str).str.strip().str.upper()


    # Filtra com base na variável (coluna cd_prova) e cor
    filtro = (
        (df_csv["cd_prova"] == variavel) &
        (df_csv["cor"] == cor)
    )

    df_filtrado = df_csv.loc[filtro, "codigo"]

   

    if not df_filtrado.empty:
        return df_filtrado.iloc[0]
    else:
        return 0


In [18]:

# Caminho para o seu arquivo CSV


def get_answer(df, codigo_prova:int, area, cor, posicao:int):
    #print(f"1{codigo_prova}, {area}, {cor}, {posicao}")
    filtros = (
        (df["CO_PROVA"] == codigo_prova) &
        (df["SG_AREA"] == area) &
        (df["CO_POSICAO"] == posicao) &
        (df["TX_COR"] == cor)
    )

    # Filtra o DataFrame pelo valor de CO_PROVA
    df_filtrado = df.loc[filtros, "TX_GABARITO"]
    #print(f"2{df_filtrado}")
    # Exibindo o resultado
    if not df_filtrado.empty:
        return df_filtrado.iloc[0]
    else:
        return ""

In [19]:
def get_item_metadado(df, codigo_prova:int, area, cor, posicao:int):
    """
    Busca metadados do item no DataFrame de ITENS_PROVA.
    
    Correção: Agora usa a área obtida do CSV como fonte primária de verdade,
    em vez de depender apenas da extração do PDF.
    """
    #print(f"1{codigo_prova}, {area}, {cor}, {posicao}")
    
    # Normalizar os valores para comparação
    cor_norm = cor.strip().upper()
    area_norm = area.strip().upper()
    
    filtros = (
        (df["CO_PROVA"] == codigo_prova) &
        (df["SG_AREA"] == area_norm) &
        (df["CO_POSICAO"] == posicao) &
        (df["TX_COR"].str.strip().str.upper() == cor_norm)
    )

    df_filtrado = df.loc[filtros]
    #print(f"2{df_filtrado}")
    
    if not df_filtrado.empty:
        return df_filtrado.iloc[0]
    else:
        return pd.Series(dtype=object)

def validate_area_from_metadata(label_extracted, metadata_row):
    """
    Validação auxiliar: compara a área extraída do PDF com a área no CSV.
    Se não coincidirem, usa a do CSV como fonte de verdade.
    
    Returns: label_correto
    """
    # Mapeamento de código de área (SG_AREA do CSV) para label esperado
    area_to_label = {
        'MT': 1,
        'LC': 2,
        'CH': 3,
        'CN': 4
    }
    
    if pd.notna(metadata_row) and 'SG_AREA' in metadata_row:
        csv_area = str(metadata_row['SG_AREA']).strip().upper()
        label_from_csv = area_to_label.get(csv_area, 0)
        
        # Se a extração do PDF deu desconhecido (0), usar a do CSV
        if label_extracted == 0:
            return label_from_csv
        
        # Se coincidirem, OK
        if label_extracted == label_from_csv:
            return label_extracted
        
        # Se não coincidirem, avisar e usar CSV (que é mais confiável)
        # print(f"AVISO: Label do PDF ({label_extracted}) != CSV ({label_from_csv}). Usando CSV.")
        return label_from_csv
    
    return label_extracted


In [20]:
def infer_area_from_question_position(question_number):
    """
    Deduz a área (label) baseado na posição da questão.
    
    ENEM Dia 1 - Estrutura fixa:
      Q1-Q45:     Linguagens (LC) → label=2
      Q46-Q90:    Ciências Humanas (CH) → label=3
      Q91-Q135:   Ciências da Natureza (CN) → label=4
      Q136-Q180:  Matemática (MT) → label=1
    
    Args:
        question_number: Número da questão (int ou str)
    
    Returns:
        dict: {'label': int, 'area': str, 'descricao': str}
    """
    try:
        q_num = int(question_number)
    except:
        return {'label': 0, 'area': 'DS', 'descricao_area': 'desconhecida'}
    
    if 1 <= q_num <= 45:
        return {'label': 2, 'area': 'LC', 'descricao_area': 'linguagens, códigos e suas tecnologias'}
    elif 46 <= q_num <= 90:
        return {'label': 3, 'area': 'CH', 'descricao_area': 'ciências humanas e suas tecnologias'}
    elif 91 <= q_num <= 135:
        return {'label': 4, 'area': 'CN', 'descricao_area': 'ciências da natureza e suas tecnologias'}
    elif 136 <= q_num <= 180:
        return {'label': 1, 'area': 'MT', 'descricao_area': 'matemática e suas tecnologias'}
    else:
        return {'label': 0, 'area': 'DS', 'descricao_area': 'desconhecida'}

def correct_area_with_csv(row):
    # Se não encontrou no CSV, usar fallback por posição
    fallback = infer_area_from_question_position(row["pos_item"])
    return fallback


In [ ]:

import os
from sklearn.model_selection import train_test_split



topicos = {
    0: 'desconhecido',
    1: 'matemática e suas tecnologias',
    2: 'linguagens, códigos e suas tecnologias',
    3: 'ciências humanas e suas tecnologias',
    4: 'ciências da natureza e suas tecnologias'
}

areas = {
    0: 'DS',
    1: 'MT',
    2: 'LC',
    3: 'CH',
    4: 'CN'
}

dias = [1,2]

anos = ["2024"]

#with os.scandir(caminho) as entradas:
    #anos = [entrada.name for entrada in entradas if entrada.is_dir()]

for ano in anos:
    enem_exam = EnemExam()
    enem_dfs = []
    
    for dia in dias:
        arquivo_prova = f"../1_data/{ano}/{ano}_PV_impresso_D{dia}.pdf"

        arquivo_dicionario = f"../1_data/{ano}/cd_provas.csv"
        microdados = pd.read_csv(arquivo_dicionario, sep=",", encoding="latin1")

        arquivo_itens_prova = f"../1_data/{ano}/ITENS_PROVA_{ano}.csv"
        # Lê o CSV usando a codificação correta
        itens_prova = pd.read_csv(arquivo_itens_prova, sep=";", encoding="latin1")

        enem_exam.read_questions(arquivo_prova)
        
        # Criar DataFrame inicial com dados extraídos do PDF
        df_initial = pd.DataFrame(data=dict( 
            enunciado=enem_exam.questions,
            ano=ano, 
            label=enem_exam.questions_topic, 
            cor_prova=enem_exam._color)
        ).assign(
            pos_item=lambda df: df["enunciado"].str.extract(r'QUESTÃO\s+(\d+)', flags=re.IGNORECASE),
            descricao_area=lambda df: df["label"].map(topicos),
            area=lambda df: df["label"].map(areas),
        )
        
        # NOVA ESTRATÉGIA: Corrigir áreas usando CSV como fonte de verdade
        # Aplicar função de correção linha por linha
        area_corrections = df_initial.apply(
            lambda row: correct_area_with_csv(row),
            axis=1,
            result_type='expand'
        )
        
        # Atualizar o DataFrame com áreas corrigidas
        df_initial['label'] = area_corrections['label']
        df_initial['area'] = area_corrections['area']
        df_initial['descricao_area'] = area_corrections['descricao_area']
        
        # Agora obter cod_prova corrigido baseado na área validada
        df_initial['cod_prova'] = df_initial["area"].map(
            lambda area: get_cod_prova_csv(area, enem_exam._color, microdados) 
            if area in areas.values() else 0
        )
        
        # Obter metadados do CSV
        enem_dfs.append(
            df_initial.pipe(
                lambda df: pd.concat([
                    df,
                    df.apply(
                        lambda row: get_item_metadado(
                            itens_prova,
                            codigo_prova=int(row["cod_prova"]),
                            area=row["area"],
                            cor=row["cor_prova"],
                            posicao=int(row["pos_item"])
                        )
                        if pd.notna(row["cod_prova"]) and pd.notna(row["pos_item"]) else pd.Series(dtype=object),
                        axis=1,
                        result_type='expand'
                    )
                ], axis=1)
            )
        )
    

    questions_df = pd.concat(enem_dfs)
    
    # VALIDAÇÃO FINAL: Verificar distribuição de áreas
    print(f"\n{ano} - Distribuição de áreas:")
    area_counts = questions_df['area'].value_counts()
    for area in ['MT', 'LC', 'CH', 'CN']:
        count = area_counts.get(area, 0)
        pct = (count / len(questions_df)) * 100
        print(f"  {area}: {count:3d} ({pct:5.1f}%)")

    train_df, test_df = train_test_split(questions_df, test_size=0.2, random_state=42)

    print(f"\nTrain: {len(train_df)} questões")
    print(f"Test: {len(test_df)} questões")

    train_df.to_json(f'../3_datasets/enem/{ano}/train/enem_questoes.json', orient='records')
    test_df.to_json(f'../3_datasets/enem/{ano}/test/enem_questoes.json', orient='records')
    
    print(f"\n✓ Datasets salvos com sucesso!")



2024 - Distribuição de áreas:
  MT:  45 ( 24.3%)
  LC:  50 ( 27.0%)
  CH:  45 ( 24.3%)
  CN:  45 ( 24.3%)

Train: 148 questões
Test: 37 questões

✓ Datasets salvos com sucesso!


In [25]:
print(enem_dfs[0].iloc[65] if len(enem_dfs) > 0 and len(enem_dfs[0]) > 65 else "Sem dados")
print("Cod_prova")
#print(get_cod_prova("CH", "AZUL", microdados))
print("Gabarito")
#get_answer(itens_prova, 1055, "CH", "AZUL", 90)


enunciado           \nQUESTÃO 61\t\nEu sentia falta do futuro. É c...
ano                                                              2024
label                                                               3
cor_prova                                                        AZUL
pos_item                                                           61
descricao_area                    ciências humanas e suas tecnologias
area                                                               CH
cod_prova                                                        1383
CO_POSICAO                                                         61
SG_AREA                                                            CH
CO_ITEM                                                        140574
TX_GABARITO                                                         B
CO_HABILIDADE                                                      11
IN_ITEM_ABAN                                                        0
TX_MOTIVO_ABAN      

In [28]:
from datasets import load_dataset, Features, Value

# Caminho para o diretorio
json_dir_path = "3_datasets/enem/"

features = Features({
    "ano": Value("int64"),
    "enunciado": Value("string"),
    "label": Value("int64"),
    "cor_prova": Value("string"),
    "pos_item": Value("string"),
    "descricao_area": Value("string"),
    "area": Value("string"),
    "cod_prova": Value("int64"),
    "CO_POSICAO": Value("float64"),
    "SG_AREA": Value("string"),
    "CO_ITEM": Value("float64"),
    "TX_GABARITO": Value("string"),
    "CO_HABILIDADE": Value("float64"),
    "IN_ITEM_ABAN": Value("float64"),
    "TX_MOTIVO_ABAN": Value("string"),
    "NU_PARAM_A": Value("float64"),
    "NU_PARAM_B": Value("float64"),
    "NU_PARAM_C": Value("float64"),
    "TX_COR": Value("string"),
    "CO_PROVA": Value("float64"),
    "TP_LINGUA": Value("string"),
    "IN_ITEM_ADAPTADO": Value("float64")
})

#dataset = load_dataset("json", data_dir="datasets", features=features)

root_path = "../3_datasets/enem"

enem_datasets = {}

for ano in anos:
    path_ano = f"{root_path}/{ano}"
    enem_datasets[ano] = load_dataset(
        "json",
        data_dir=path_ano,
        features=features
    )

# Exibir algumas amostras do dataset
print(enem_datasets)

Generating train split: 148 examples [00:00, 3692.63 examples/s]
Generating test split: 37 examples [00:00, 4231.93 examples/s]

{'2024': DatasetDict({
    train: Dataset({
        features: ['ano', 'enunciado', 'label', 'cor_prova', 'pos_item', 'descricao_area', 'area', 'cod_prova', 'CO_POSICAO', 'SG_AREA', 'CO_ITEM', 'TX_GABARITO', 'CO_HABILIDADE', 'IN_ITEM_ABAN', 'TX_MOTIVO_ABAN', 'NU_PARAM_A', 'NU_PARAM_B', 'NU_PARAM_C', 'TX_COR', 'CO_PROVA', 'TP_LINGUA', 'IN_ITEM_ADAPTADO'],
        num_rows: 148
    })
    test: Dataset({
        features: ['ano', 'enunciado', 'label', 'cor_prova', 'pos_item', 'descricao_area', 'area', 'cod_prova', 'CO_POSICAO', 'SG_AREA', 'CO_ITEM', 'TX_GABARITO', 'CO_HABILIDADE', 'IN_ITEM_ABAN', 'TX_MOTIVO_ABAN', 'NU_PARAM_A', 'NU_PARAM_B', 'NU_PARAM_C', 'TX_COR', 'CO_PROVA', 'TP_LINGUA', 'IN_ITEM_ADAPTADO'],
        num_rows: 37
    })
})}


In [ ]:
# import pandas as pd

# for ano in anos:
#     enem_datasets[ano].set_format(type="pandas")
#     df = enem_datasets[ano]["train"][:]

#     # Filtra o DataFrame pelo valor de CO_PROVA
#     df_filtrado = df[df["CO_POSICAO"] == 98]

#     #Imprime as linhas filtradas
#     for index, row in df_filtrado.iterrows():
#         print(f"Questão: {row['pos_item']}")
#         print(f"Ano: {row['ano']}")
#         print(f"Cor caderno: {row['cor_prova']}")
#         print(f"Área: {row['area']}")
#         print(f"Descrição: {row['descricao_area']}")
#         print(f"Enunciado: {row['enunciado']}")
#         print(f"Gabarito: {row['TX_GABARITO']}")
#         print(f"Dificuldade: {row['NU_PARAM_B']}")
#         print(50 * "-")

In [29]:
# ===== VALIDAÇÃO DOS RESULTADOS =====
print("\n" + "="*70)
print("VALIDAÇÃO - Exemplos de Questões Corrigidas")
print("="*70)

# Verificar Q175 específicamente
if len(enem_dfs) > 0:
    df_resultado = questions_df
    
    q175 = df_resultado[df_resultado['pos_item'] == '175']
    
    if not q175.empty:
        print("\nQ175 - Verificação Específica:")
        row = q175.iloc[0]
        print(f"  area: {row['area']}")
        print(f"  label: {row['label']}")
        print(f"  descricao_area: {row['descricao_area']}")
        print(f"  cod_prova: {row['cod_prova']}")
        print(f"  cor_prova: {row['cor_prova']}")
        
        # Verificar contra CSV
        csv_check = itens_prova[
            (itens_prova['CO_POSICAO'] == 175) & 
            (itens_prova['TX_COR'].str.upper() == row['cor_prova'].upper())
        ]
        
        if not csv_check.empty:
            csv_area = csv_check.iloc[0]['SG_AREA']
            csv_cod = csv_check.iloc[0]['CO_PROVA']
            match = "✓ CORRETO" if row['area'] == csv_area else "✗ ERRO"
            print(f"  {match} - CSV diz: area={csv_area}, cod_prova={csv_cod}")
    
    print("\nExemplos de Questões por Área:")
    for area in ['MT', 'LC', 'CH', 'CN']:
        sample = df_resultado[df_resultado['area'] == area].iloc[0:2]
        if not sample.empty:
            for _, row in sample.iterrows():
                print(f"  Q{row['pos_item']}: {area} (label={row['label']})")

print("\n" + "="*70)



VALIDAÇÃO - Exemplos de Questões Corrigidas

Q175 - Verificação Específica:
  area: MT
  label: 1
  descricao_area: matemática e suas tecnologias
  cod_prova: 1407
  cor_prova: AZUL
  ✓ CORRETO - CSV diz: area=MT, cod_prova=1407

Exemplos de Questões por Área:
  Q136: MT (label=1)
  Q137: MT (label=1)
  Q01: LC (label=2)
  Q02: LC (label=2)
  Q46: CH (label=3)
  Q47: CH (label=3)
  Q91: CN (label=4)
  Q92: CN (label=4)

